# Sprint 1 — Fundamentos de IA Generativa (versão revisada)

> Esta versão inclui **execução real de prompts** com API (quando chave disponível) e fallback local/mock para permitir demonstração completa de input/output.

## Integrantes do grupo:
1. Lucca Phelipe Masini (RM 564121)
2. Luiz Henrique Poss (RM562177)
3. Igor Paixão Sarak (RM 563726)
4. Bernardo Braga Perobeli (RM 562468)
5. Felipe Stefani Honorato (RM 563380)

## 0. Setup
Defina as chaves no Colab (`GROQ_API_KEY`) e, opcionalmente, endpoints locais (Ollama).



In [ ]:
import os
import json
import pandas as pd
from dataclasses import dataclass
from typing import Dict, List

USE_REAL_API = bool(os.getenv("GROQ_API_KEY"))
print("USE_REAL_API:", USE_REAL_API)




USE_REAL_API: False


In [ ]:
if USE_REAL_API:
    from openai import OpenAI
    client = OpenAI(api_key=os.getenv("GROQ_API_KEY"), base_url="https://api.groq.com/openai/v1")




## 1. Comparativo de arquiteturas e seleção



In [ ]:
models = pd.DataFrame([
    {"modelo":"gpt-4.1-mini","arquitetura":"LLM","multilingue":"alto","execucao":"API","custo_relativo":"médio"},
    {"modelo":"llama-3.1-8b-instruct","arquitetura":"LLM","multilingue":"médio","execucao":"local/API","custo_relativo":"baixo"},
    {"modelo":"gemma-2-9b-it","arquitetura":"LLM","multilingue":"médio","execucao":"local/API","custo_relativo":"baixo"},
    {"modelo":"qwen2.5-7b-instruct","arquitetura":"LLM","multilingue":"médio","execucao":"local/API","custo_relativo":"baixo"},
    {"modelo":"stable-diffusion-xl","arquitetura":"Difusão","multilingue":"n/a","execucao":"local/API","custo_relativo":"médio"},
    {"modelo":"VAE-tabular","arquitetura":"VAE","multilingue":"n/a","execucao":"local","custo_relativo":"baixo"}
])
models




,modelo,arquitetura,multilingue,execucao,custo_relativo
0,gpt-4.1-mini,LLM,alto,API,médio
1,llama-3.1-8b-instruct,LLM,médio,local/API,baixo
2,gemma-2-9b-it,LLM,médio,local/API,baixo
3,qwen2.5-7b-instruct,LLM,médio,local/API,baixo
4,stable-diffusion-xl,Difusão,n/a,local/API,médio
5,VAE-tabular,VAE,n/a,local,baixo


## 2. Biblioteca de prompts do domínio



In [ ]:
prompt_library = {
"zero_shot_diag_v1": "Classifique o estado do motor (normal/alerta/crítico), explique causa e proponha 2 ações.",
"few_shot_os_v2": "Exemplo A: vib alta+temp alta=>desalinhamento. Exemplo B: corrente alta=>sobrecorrente. Agora resolva o caso.",
"cot_analysis_v1": "Raciocine em etapas, mas retorne apenas JSON final com diagnóstico e ação.",
"roleplay_engineer_v1": "Atue como engenheiro de manutenção industrial bilíngue PT/EN, preservando unidades SI."
}
prompt_library




{'zero_shot_diag_v1': 'Classifique o estado do motor (normal/alerta/crítico), explique causa e proponha 2 ações.',
 'few_shot_os_v2': 'Exemplo A: vib alta+temp alta=>desalinhamento. Exemplo B: corrente alta=>sobrecorrente. Agora resolva o caso.',
 'cot_analysis_v1': 'Raciocine em etapas, mas retorne apenas JSON final com diagnóstico e ação.',
 'roleplay_engineer_v1': 'Atue como engenheiro de manutenção industrial bilíngue PT/EN, preservando unidades SI.'}

## 3. Harness de teste (input/output real)



In [ ]:
@dataclass
class TestCase:
    id: str
    sensor_json: Dict
    expected_fault: str

TEST_CASES = [
    TestCase("tc1", {"temp_c":102, "vibration_mm_s":9.8, "current_a":78, "rpm":1780}, "Sobreaquecimento com desalinhamento"),
    TestCase("tc2", {"temp_c":82, "vibration_mm_s":6.1, "current_a":92, "rpm":1760}, "Sobrecorrente por atrito mecânico"),
    TestCase("tc3", {"temp_c":61, "vibration_mm_s":3.3, "current_a":41, "rpm":1800}, "Operação normal"),
]

SYSTEM_MSG = "Você é engenheiro de manutenção de motores elétricos industriais. Responda em JSON com campos: diagnostico, severidade, acoes."

def mock_inference(sensor_json: Dict) -> str:
    t, v, c = sensor_json["temp_c"], sensor_json["vibration_mm_s"], sensor_json["current_a"]
    if t > 95 and v > 8:
        fault, sev = "Sobreaquecimento com desalinhamento", "alta"
    elif c > 85:
        fault, sev = "Sobrecorrente por atrito mecânico", "alta"
    else:
        fault, sev = "Operação normal", "baixa"
    return json.dumps({"diagnostico": fault, "severidade": sev, "acoes": ["inspecionar", "monitorar"]}, ensure_ascii=False)


def run_groq(model: str, prompt: str) -> str:
    response = client.chat.completions.create(
        model=model,
        temperature=0.2,
        top_p=0.9,
        messages=[
            {"role":"system","content":SYSTEM_MSG},
            {"role":"user","content":prompt}
        ]
    )
    return response.choices[0].message.content




In [ ]:
def parse_diag(text: str) -> str:
    try:
        return json.loads(text).get("diagnostico", "")
    except Exception:
        return text

results = []
models_to_test = ["llama-3.3-70b-versatile", "mixtral-8x7b-32768"] if USE_REAL_API else ["mock-llm"]
techniques = ["zero_shot_diag_v1", "few_shot_os_v2", "cot_analysis_v1", "roleplay_engineer_v1"]

for m in models_to_test:
    for tc in TEST_CASES:
        for tech in techniques:
            user_prompt = f"""{prompt_library[tech]}
Dados: {tc.sensor_json}"""
            output = run_groq(m, user_prompt) if USE_REAL_API else mock_inference(tc.sensor_json)
            pred = parse_diag(output)
            correct = int(tc.expected_fault.lower() in pred.lower())
            results.append({
                "model": m,
                "case_id": tc.id,
                "technique": tech,
                "input": tc.sensor_json,
                "output": output,
                "expected": tc.expected_fault,
                "match_expected": correct,
            })

df_results = pd.DataFrame(results)
df_results.head(8)

,model,case_id,technique,input,output,expected,match_expected
0,mock-llm,tc1,zero_shot_diag_v1,"{'temp_c': 102, 'vibration_mm_s': 9.8, 'curren...","{""diagnostico"": ""Sobreaquecimento com desalinh...",Sobreaquecimento com desalinhamento,1
1,mock-llm,tc1,few_shot_os_v2,"{'temp_c': 102, 'vibration_mm_s': 9.8, 'curren...","{""diagnostico"": ""Sobreaquecimento com desalinh...",Sobreaquecimento com desalinhamento,1
2,mock-llm,tc1,cot_analysis_v1,"{'temp_c': 102, 'vibration_mm_s': 9.8, 'curren...","{""diagnostico"": ""Sobreaquecimento com desalinh...",Sobreaquecimento com desalinhamento,1
3,mock-llm,tc1,roleplay_engineer_v1,"{'temp_c': 102, 'vibration_mm_s': 9.8, 'curren...","{""diagnostico"": ""Sobreaquecimento com desalinh...",Sobreaquecimento com desalinhamento,1
4,mock-llm,tc2,zero_shot_diag_v1,"{'temp_c': 82, 'vibration_mm_s': 6.1, 'current...","{""diagnostico"": ""Sobrecorrente por atrito mecâ...",Sobrecorrente por atrito mecânico,1
5,mock-llm,tc2,few_shot_os_v2,"{'temp_c': 82, 'vibration_mm_s': 6.1, 'current...","{""diagnostico"": ""Sobrecorrente por atrito mecâ...",Sobrecorrente por atrito mecânico,1
6,mock-llm,tc2,cot_analysis_v1,"{'temp_c': 82, 'vibration_mm_s': 6.1, 'current...","{""diagnostico"": ""Sobrecorrente por atrito mecâ...",Sobrecorrente por atrito mecânico,1
7,mock-llm,tc2,roleplay_engineer_v1,"{'temp_c': 82, 'vibration_mm_s': 6.1, 'current...","{""diagnostico"": ""Sobrecorrente por atrito mecâ...",Sobrecorrente por atrito mecânico,1


In [ ]:
summary = (
    df_results.groupby(["model", "technique"], as_index=False)["match_expected"]
    .mean()
    .rename(columns={"match_expected":"factual_accuracy"})
)
summary




,model,technique,factual_accuracy
0,mock-llm,cot_analysis_v1,1.0
1,mock-llm,few_shot_os_v2,1.0
2,mock-llm,roleplay_engineer_v1,1.0
3,mock-llm,zero_shot_diag_v1,1.0


## 4. Geração de dados sintéticos (200 registros)



In [ ]:
df = pd.read_csv('synthetic_motor_records.csv')
print("Registros:", len(df))
df.sample(3, random_state=7)




Registros: 200


,record_id,timestamp,motor_id,rpm,temperature_c,vibration_mm_s,current_a,fault_label,severity,work_order,maintenance_report,model,temperature,top_p,prompt_template,validation_status
86,87,2026-01-22T12:00:00,MTR-1011,3138,81.0,5.08,94.7,Sobrecorrente por atrito mecânico,alta,WO-20260086,Inspeção do motor MTR-1011: temperatura 81.0°C...,gemma-2-9b-it,0.4,0.90,roleplay_engineer_v1,aprovado
120,121,2026-01-31T00:00:00,MTR-1020,2377,109.8,9.64,49.7,Sobreaquecimento com desalinhamento,alta,WO-20260120,Inspeção do motor MTR-1020: temperatura 109.8°...,gpt-4.1-mini,0.7,0.95,zero_shot_diag_v1,aprovado
22,23,2026-01-06T12:00:00,MTR-1022,2120,49.2,1.44,59.3,Operação normal,baixa,WO-20260022,Inspeção do motor MTR-1022: temperatura 49.2°C...,gemma-2-9b-it,0.2,0.85,zero_shot_diag_v1,aprovado


In [ ]:
checks = {
 "temp_range_ok": float(df["temperature_c"].between(30,130).mean()),
 "vibration_range_ok": float(df["vibration_mm_s"].between(0.5,20).mean()),
 "approved_ratio": float((df["validation_status"]=="aprovado").mean()),
}
checks




{'temp_range_ok': 1.0, 'vibration_range_ok': 1.0, 'approved_ratio': 0.99}

## 5. Export de evidências



In [ ]:
from pathlib import Path

Path('prompt_eval_results.csv').write_text(df_results.to_csv(index=False), encoding='utf-8')
Path('prompt_eval_summary.csv').write_text(summary.to_csv(index=False), encoding='utf-8')
print('Arquivos de avaliação salvos: prompt_eval_results.csv, prompt_eval_summary.csv')

Arquivos de avaliação salvos: prompt_eval_results.csv, prompt_eval_summary.csv
